In [ ]:
import time, numpy as np
import matplotlib.pyplot as plt
from skopt.space import Real
from skopt import gp_minimize
from skopt.utils import use_named_args
import mph
import os
import pandas as pd

n_calls=50
CSV_PATH = f"BO_history_{time.strftime('%Y%m%d_%H%M%S')}_{n_calls}.csv"
def log_csv(row: dict, path=CSV_PATH): pd.DataFrame([row]).to_csv( path, mode="a", header=not os.path.exists(path), index=False)
    
# --- Arranque COMSOL ---
try:
    client.clear()
except:
    pass

client = mph.start()
model  = client.load('PBMR3.mph')

# IMPORTANTE: que sea el dataset del estudio que resuelves (Study 2)
DATASET = "Study 5//Solution 33"
PENALTY = 1e6

def run_study5(model):
    model.solve('Study 5')

space = [
    Real(290, 380, name='T0'),
    Real(1.5, 4,   name='P0'),
    Real(500, 2000, name='F'),  # mmol/h
]

def set_inputs(model, T0_C, P0_bar, F_mmol_h):
    model.parameter('T0',      f'{T0_C} [degC]')
    model.parameter('P0',      f'{P0_bar} [bar]')
    model.parameter('F_C2H60', f'{F_mmol_h} [mmol/h]')
    model.parameter('R_C2H6_O2', '2')
    model.parameter('y_O2_1', '45')
    model.parameter('y_O2_2', '25')
    model.parameter('y_O2_3', '30')

# Helpers
def ev(expr, dataset):
    return float(model.evaluate(expr, dataset=dataset))

def ev_field(expr, dataset):
    return np.asarray(model.evaluate(expr, dataset=dataset), float).ravel()

HISTORY = []

def read_metrics_all(model, dataset):
    eps = 1e-12

    # --- Métricas principales (C2H4) ---
    FA_in  = -ev("tds.in1.nmflow_C_C2H6", dataset)
    FA_out =  ev("tds.out1.nmflow_C_C2H6",  dataset)
    FP_out =  ev("tds3.out1.nmflow_C_C2H4", dataset)

    X = 100*(FA_in-FA_out)/max(FA_in, eps)
    S = 100*FP_out/max(FA_in-FA_out, eps)
    Y = 100*FP_out/max(FA_in, eps)

    # --- Caída de presión ---
    Pin  = ev("spf.inl1.pAverage",  dataset)
    Pout = ev("spf.out1.pAverage",  dataset)
    dP_Pa   = Pin - Pout
    dP_mbar = dP_Pa / 100.0

    # --- Side products (tus expresiones) ---
    S_CO  = ev("100*tds5.out1.nmflow_C_CO/(2*(abs(tds.in1.nmflow_C_C2H6)-tds.out1.nmflow_C_C2H6))", dataset)
    Y_CO  = ev("100*tds5.out1.nmflow_C_CO/(2*abs(tds.in1.nmflow_C_C2H6))", dataset)

    S_CO2 = ev("100*tds6.out1.nmflow_C_CO2/(2*(abs(tds.in1.nmflow_C_C2H6)-tds.out1.nmflow_C_C2H6))", dataset)
    Y_CO2 = ev("100*tds6.out1.nmflow_C_CO2/(2*abs(tds.in1.nmflow_C_C2H6))", dataset)

    # AcOH (C2): fórmula corregida sin /2
    S_AcOH = ev("100*tds7.out1.nmflow_C_AcOH/(abs(tds.in1.nmflow_C_C2H6)-tds.out1.nmflow_C_C2H6)", dataset)
    Y_AcOH = ev("100*tds7.out1.nmflow_C_AcOH/(abs(tds.in1.nmflow_C_C2H6))", dataset)

    # --- Hot spot: Tmax + ubicación + U(Tmax) ---
    x = ev_field('x', dataset)
    y = ev_field('y', dataset)
    T = ev_field('T', dataset)
    U = ev_field('spf.U', dataset)

    i = int(np.nanargmax(T))
    Tmax_C = float(T[i] - 273.15)
    xT, yT = float(x[i]), float(y[i])
    U_Tmax = float(U[i])

    # --- Space velocity: WHSV y W/F desde parámetros (no dependen del dataset) ---
    F_mol_s  = model.parameter('F_C2H60', evaluate=True)  # mol/s
    M_kg_mol = model.parameter('M_C2H6',  evaluate=True)  # kg/mol
    mcat_kg  = model.parameter('m_cat',   evaluate=True)  # kg

    WHSV_1_h = (F_mol_s * M_kg_mol * 3600.0) / mcat_kg     # 1/h
    W_over_F = (mcat_kg * 1000.0) / (F_mol_s * 3600.0)     # gcat·h/mol

    return {
        "X": X, "S": S, "Y": Y,
        "dP_mbar": dP_mbar,
        "S_CO": S_CO, "Y_CO": Y_CO,
        "S_CO2": S_CO2, "Y_CO2": Y_CO2,
        "S_AcOH": S_AcOH, "Y_AcOH": Y_AcOH,
        "Tmax_C": Tmax_C, "xT": xT, "yT": yT, "U_Tmax": U_Tmax,
        "WHSV_1_h": float(WHSV_1_h), "W_over_F": float(W_over_F),
    }

@use_named_args(space)
def objective(T0, P0, F):
    try:
        set_inputs(model, T0, P0, F)
        t0 = time.time()
        run_study5(model)

        m = read_metrics_all(model, DATASET)
        t = time.time() - t0

        print(
            f"[OK] Y={m['Y']:.3f}% S={m['S']:.3f}% X={m['X']:.3f}% | "
            f"T0={T0:.2f}°C P0={P0:.3f}bar F={F:.1f} mmol/h | t={t:.2f}s\n"
            f"     ΔP={m['dP_mbar']:.3f} mbar | Tmax={m['Tmax_C']:.2f}°C @({m['xT']:.3e},{m['yT']:.3e}) | U@Tmax={m['U_Tmax']:.3e} m/s\n"
            f"     WHSV={m['WHSV_1_h']:.3f} 1/h | W/F={m['W_over_F']:.3f} gcat·h/mol | "
            f"S_CO={m['S_CO']:.3f}% Y_CO={m['Y_CO']:.3f}% | "
            f"S_CO2={m['S_CO2']:.3f}% Y_CO2={m['Y_CO2']:.3f}% | "
            f"S_AcOH={m['S_AcOH']:.3f}% Y_AcOH={m['Y_AcOH']:.3f}%"
        )

        row = {
            **m,
            "T0": float(T0), "P0": float(P0), "F": float(F),
            "t_s": float(t),
            "status": "OK"
        }

        HISTORY.append(row)
        log_csv(row)

        return -m["Y"]
    except Exception as e:
        print(f"[FAIL] {e}")
        log_csv({"T0": float(T0), "P0": float(P0), "F": float(F), "status": "FAIL", "error": str(e)})
        return PENALTY

res = gp_minimize(
    objective, space,
    n_calls=n_calls, n_initial_points=5,
    acq_func="EI", random_state=11
)

best_Y = -res.fun
best_T0, best_P0, best_F = res.x
print("\n===== RESULTADO FINAL =====")
print(f"Mejor Y ≈ {best_Y:.3f}%  |  T0={best_T0:.2f}°C  P0={best_P0:.3f}bar  F={best_F:.2f} mmol/h")

# TOP 10 por Y
top = sorted(HISTORY, key=lambda r: r["Y"], reverse=True)[:10]
print("\n===== TOP 10 (por Rendimiento Y) =====")
for i, r in enumerate(top, 1):
    print(f"{i:02d}) Y={r['Y']:.3f}% S={r['S']:.2f}% X={r['X']:.2f}% | "
          f"T0={r['T0']:.2f}°C P0={r['P0']:.3f}bar F={r['F']:.1f} | "
          f"ΔP={r['dP_mbar']:.3f} mbar Tmax={r['Tmax_C']:.2f}°C U@Tmax={r['U_Tmax']:.3e} | "
          f"SCO={r['S_CO']:.2f}% SCO2={r['S_CO2']:.2f}% SAcOH={r['S_AcOH']:.2f}%")

def plot_bo_conv(history, key="Y", ylabel="Rendimiento Y (%)", title="Convergencia BO", save=None):
    Y = np.asarray([r[key] for r in history], float); it = np.arange(1, Y.size+1)
    best = np.maximum.accumulate(Y); i = int(np.nanargmax(Y))
    with plt.rc_context({"font.family":"serif","font.serif":"Times New Roman","font.size":12,"figure.dpi":150,
                         "axes.edgecolor":"0.25","axes.labelcolor":"0.15","xtick.color":"0.25","ytick.color":"0.25"}):
        fig, ax = plt.subplots(figsize=(7,5))
        ax.scatter(it, Y, s=34, c="#B24A3A", alpha=.55, label="Evaluaciones")
        ax.plot(it, best, "k-", lw=2.4, label="Mejor acumulado")
        ax.plot(it[i], Y[i], "^", ms=11, mfc="#B24A3A", mec="#B24A3A", ls="none", label="Óptimo")
        ax.set(xlabel="Iteración", ylabel=ylabel, title=title); ax.grid(True, alpha=.18)
        ax.tick_params(length=4, width=1, direction="out"); ax.margins(x=.02, y=.05)
        ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02,.5))
        fig.tight_layout()
        if save: fig.savefig(save, bbox_inches="tight", dpi=600)
        plt.show()
plot_bo_conv(HISTORY, ylabel="Rendimiento Y (%)", title="Convergencia BO (T0, P0, F)", save="bo_convergencia_Final.png")
#plt.figure(figsize=(7,4))
#plt.plot(range(1, len(Y_list)+1), best_so_far, lw=2)
#plt.scatter(range(1, len(Y_list)+1), Y_list, s=20, alpha=0.4)
#plt.xlabel("Iteración")
#plt.ylabel("Mejor rendimiento Y (%)")
#plt.title("Convergencia BO (mejor Y vs. iteraciones)")
#plt.grid(True, alpha=0.5)
#plt.tight_layout()
#plt.show()